In [ ]:
import turtle
import re

def create_turtle_controller(config):
    t= turtle.Turtle()
    current_color= "black"
    distance_scale= config["scale"]
    valid_command_count= 0
    invalid_command_count= 0
    command_history= []

    t.shape(config["shape"])

    scale_dist= lambda d: float(d)*distance_scale

    def draw_recursive_spiral(length, depth):
        if depth <= 0 or length <= 0:
            return
        t.forward(length)
        t.right(90)
        draw_recursive_spiral(length-5, depth-1)

    def execute(command_text):
        nonlocal current_color, valid_command_count, invalid_command_count, command_history

        tokens= command_text.strip().split(";")
        token_iterator= iter(tokens)

        while True:
            try:
                cmd= next(token_iterator).strip()
            except StopIteration:
                break

            try:
                status_pattern= re.compile(r"^(status|report)$")
                move_pattern= re.compile(r"^([FLR])(\d+)$")
                color_pattern= re.compile(r"^C([a-zA-Z]+)$")
                shape_pattern= re.compile(r"^Shape\s+(\w+)$")
                fill_pattern= re.compile(r"^Fill\s+(on|off)$")
                sq_pattern= re.compile(r"^SQ(\d+)$")
                poly_pattern= re.compile(r"^POLY(\d+):(\d+)$")
                spiral_pattern= re.compile(r"^SPIRAL(\d+):(\d+)$")

                if status_pattern.search(cmd):
                    print(f"cureent color: {current_color}, scale: {distance_scale}, total valid: {valid_command_count}, total invald: {invalid_command_count}")

                elif move_pattern.search(cmd):
                    match= move_pattern.search(cmd)
                    action= match.group(1)
                    val= int(match.group(2))
                    if action == "F":
                        t.forward(scale_dist(val))
                    elif action == "L":
                        t.left(val)
                    elif action == "R":
                        t.right(val)

                elif color_pattern.search(cmd):
                    match= color_pattern.search(cmd)
                    t.color(match.group(1))
                    current_color = match.group(1)

                elif shape_pattern.search(cmd):
                    match= shape_pattern.search(cmd)
                    t.shape(match.group(1))

                elif fill_pattern.search(cmd):
                    match= fill_pattern.search(cmd)
                    if match.group(1) == "on":
                        t.begin_fill()
                    else:
                        t.end_fill()

                elif sq_pattern.search(cmd):
                    match= sq_pattern.search(cmd)
                    side= scale_dist(match.group(1))
                    for i in range(4):
                        t.forward(side)
                        t.right(90)

                elif poly_pattern.search(cmd):
                    match= poly_pattern.search(cmd)
                    sides= int(match.group(1))
                    length= scale_dist(match.group(2))
                    for i in range(sides):
                        t.forward(length)
                        t.right(360/sides)

                elif spiral_pattern.search(cmd):
                    match= spiral_pattern.search(cmd)
                    draw_recursive_spiral(scale_dist(match.group(1)), int(match.group(2)))

                else:
                    raise ValueError(f"invaled command format: {cmd}")

                valid_command_count+= 1
                command_history.append(cmd)

            except Exception as eror:
                invalid_command_count+= 1
                print(f"eror: {eror}")

    def search_history(pattern):
        regex= re.compile(pattern)
        results= []
        for cmd in command_history:
            if regex.search(cmd):
                results.append(cmd)
        return results

    return execute, search_history

turtle.setup(width=800, height=600)

run1, find1= create_turtle_controller({"scale": 1.0, "shape": "turtle"})
run2, find2= create_turtle_controller({"scale": 0.5, "shape": "classic"})

while True:
    user_input= input("Enter command (or 'exit' to quit): ")
    if user_input.strip() == "exit":
        break

    which_turtle= input("Which turtle do you wanna run (1 or 2): ")
    if which_turtle.strip() == "1":
        run1(user_input)
    elif which_turtle.strip() == "2":
        run2(user_input)
    else:
        print("you chose an invalid turtle")

turtle.done()

TclError: no display name and no $DISPLAY environment variable